<a href="https://colab.research.google.com/github/MurphyKlein/CS4782_final_project/blob/main/notebook/lin.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

BASE_DIR = "/content/drive/MyDrive/DL_Final_Project"

# ─────────────────────────────────────────────
# 1. WEATHER DATASET  (mpi_roof_2025.csv)
# ─────────────────────────────────────────────
WEATHER_DIR  = f"{BASE_DIR}/weather_data"
WEATHER_FILE = os.path.join(WEATHER_DIR, "mpi_roof_2025.csv")

def load_weather(filepath):
    df = pd.read_csv(filepath, encoding='unicode_escape')
    print(f"[Weather] Raw shape: {df.shape}")

    # drop any datetime / string columns
    date_cols = [c for c in df.columns if 'date' in c.lower() or 'time' in c.lower()]
    if date_cols:
        df = df.drop(columns=date_cols)

    df = df.select_dtypes(include=[np.number])

    # Keep all weather data (zeros allowed)
    df = df.dropna()
    print(f"[Weather] Final shape: {df.shape}")
    return df

weather_df = load_weather(WEATHER_FILE)

# ─────────────────────────────────────────────
# 2. ELECTRICITY DATASET  (LD2011_2014.txt)
# ─────────────────────────────────────────────
ELECTRICITY_DIR  = f"{BASE_DIR}/electricity_data"
ELECTRICITY_FILE = os.path.join(ELECTRICITY_DIR, "LD2011_2014.txt")

def load_electricity(filepath):
    print("[Electricity] Loading... (this may take a moment)")
    df = pd.read_csv(filepath, sep=';', decimal=',', index_col=0, parse_dates=True)
    print(f"[Electricity] Raw shape: {df.shape}")

    # resample 15-min → hourly
    df = df.resample('1H').mean()

    # Drop columns that contain any 0s (for Electricity only)
    cols_with_zeros = df.columns[(df == 0).any()]
    print(f"[Electricity] Dropping {len(cols_with_zeros)} columns containing zeros.")
    df = df.drop(columns=cols_with_zeros)

    df = df.dropna()
    print(f"[Electricity] Final shape: {df.shape}")
    return df

electricity_df = load_electricity(ELECTRICITY_FILE)

# ─────────────────────────────────────────────
# 3. SPLIT + SCALE
# ─────────────────────────────────────────────
def split_and_scale(df, train_ratio=0.7, val_ratio=0.1):
    n       = len(df)
    n_train = int(n * train_ratio)
    n_val   = int(n * val_ratio)

    train = df.iloc[:n_train].values.astype(np.float32)
    val   = df.iloc[n_train : n_train + n_val].values.astype(np.float32)
    test  = df.iloc[n_train + n_val:].values.astype(np.float32)

    scaler = StandardScaler().fit(train)
    return scaler.transform(train), scaler.transform(val), scaler.transform(test), scaler

print("\n── Weather splits ──")
w_train, w_val, w_test, w_scaler = split_and_scale(weather_df)
print(f"  train {w_train.shape}  val {w_val.shape}  test {w_test.shape}")

print("\n── Electricity splits ──")
e_train, e_val, e_test, e_scaler = split_and_scale(electricity_df)
print(f"  train {e_train.shape}  val {e_val.shape}  test {e_test.shape}")

print("\nAll done — data ready for DataLoader.")

[Weather] Raw shape: (52560, 22)
[Weather] Final shape: (52560, 21)
[Electricity] Loading... (this may take a moment)
[Electricity] Raw shape: (140256, 370)


/tmp/ipykernel_5096/1508700158.py:44: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df = df.resample('1H').mean()


[Electricity] Dropping 280 columns containing zeros.
[Electricity] Final shape: (35065, 90)

── Weather splits ──
  train (36792, 21)  val (5256, 21)  test (10512, 21)

── Electricity splits ──
  train (24545, 90)  val (3506, 90)  test (7014, 90)

All done — data ready for DataLoader.


In [ ]:
print("--- Weather Dataset Preview ---")
display(weather_df.head())

print("\n--- Electricity Dataset Preview ---")
display(electricity_df.head())

--- Weather Dataset Preview ---


,p (mbar),T (degC),Tpot (K),Tdew (degC),rh (%),VPmax (mbar),VPact (mbar),VPdef (mbar),sh (g/kg),H2OC (mmol/mol),...,wv (m/s),max. wv (m/s),wd (deg),rain (mm),raining (s),SWDR (W/m²),PAR (µmol/m²/s),max. PAR (µmol/m²/s),Tlog (degC),CO2 (ppm)
0,997.30,-1.56,271.80,-2.97,90.0,5.45,4.90,0.54,3.06,4.92,...,1.61,3.87,148.9,0.0,0.0,0.0,0.0,0.0,6.24,456.9
1,997.05,-0.90,272.48,-2.66,87.8,5.72,5.02,0.70,3.14,5.03,...,1.71,3.05,145.6,0.0,0.0,0.0,0.0,0.0,6.32,451.7
2,996.96,-0.51,272.88,-2.44,86.7,5.89,5.10,0.78,3.19,5.12,...,1.61,3.31,146.9,0.0,0.0,0.0,0.0,0.0,6.46,451.0
3,996.77,-0.27,273.14,-2.26,86.3,5.99,5.17,0.82,3.23,5.19,...,1.86,2.93,137.0,0.0,0.0,0.0,0.0,0.0,6.63,450.4
4,996.62,-0.09,273.33,-2.14,86.0,6.07,5.22,0.85,3.26,5.24,...,1.25,2.23,131.3,0.0,0.0,0.0,0.0,0.0,6.82,450.4



--- Electricity Dataset Preview ---


,MT_156,MT_158,MT_159,MT_166,MT_168,MT_169,MT_171,MT_172,MT_176,MT_180,...,MT_318,MT_319,MT_320,MT_321,MT_323,MT_324,MT_326,MT_327,MT_329,MT_330
2011-01-01 00:00:00,68.203369,38.787879,20.363985,831.135531,56.023326,22.983670,146.988922,80.792860,41.150912,93.291732,...,92.685753,47.336090,60.661464,56.506239,987.520325,240.633803,138.395677,270.557342,130.117647,45.708333
2011-01-01 01:00:00,68.359326,38.673128,19.887452,769.752747,59.224041,19.736311,151.540785,76.634496,41.482587,86.271451,...,98.742331,47.188423,59.035674,59.054622,982.408537,230.792254,127.892768,285.032154,133.305147,44.612500
2011-01-01 02:00:00,68.359326,39.010695,21.087165,783.516484,59.503386,19.590778,143.202417,75.650685,38.243781,85.881435,...,96.947853,46.743946,56.665273,61.923224,939.695122,225.510563,130.701372,272.966238,126.316176,46.332812
2011-01-01 03:00:00,68.203369,39.004011,20.129310,778.021978,58.941874,19.736311,140.158610,74.034869,40.485075,83.541342,...,93.624744,45.118133,57.083333,59.054622,957.987805,226.390845,131.954489,264.927653,127.047794,45.865625
2011-01-01 04:00:00,68.359326,38.338904,21.087165,808.269231,58.659707,19.159942,140.929003,76.637609,39.987562,89.781591,...,97.972904,44.527466,54.990245,62.307105,967.134146,225.070423,132.883416,260.096463,128.150735,43.675000


In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 1: Imports & device
# ─────────────────────────────────────────────────────────────────
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

Using device: cuda


In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 2: Dataset  — sliding window, returns (x, y) pairs
# ─────────────────────────────────────────────────────────────────
class TimeSeriesDataset(Dataset):
    """
    Returns (x, y) where:
        x : (seq_len,  n_channels)   look-back window
        y : (pred_len, n_channels)   forecast horizon
    """
    def __init__(self, data: np.ndarray, seq_len: int, pred_len: int):
        self.data     = torch.tensor(data, dtype=torch.float32)
        self.seq_len  = seq_len
        self.pred_len = pred_len

    def __len__(self):
        return len(self.data) - self.seq_len - self.pred_len + 1

    def __getitem__(self, idx):
        x = self.data[idx            : idx + self.seq_len]
        y = self.data[idx+self.seq_len : idx + self.seq_len + self.pred_len]
        return x, y


def make_loaders(train, val, test, seq_len, pred_len, batch_size=32):
    def loader(arr, shuffle):
        ds = TimeSeriesDataset(arr, seq_len, pred_len)
        return DataLoader(ds, batch_size=batch_size, shuffle=shuffle,
                          num_workers=2, pin_memory=True, drop_last=False)
    return loader(train, True), loader(val, False), loader(test, False)

In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 3: PatchTST model
# ─────────────────────────────────────────────────────────────────
class PatchEmbedding(nn.Module):
    """Segment a univariate series into patches + linear projection."""
    def __init__(self, patch_len, stride, d_model, seq_len):
        super().__init__()
        self.patch_len = patch_len
        self.stride    = stride
        self.n_patches = (seq_len - patch_len) // stride + 2   # +2 from paper's padding
        self.proj      = nn.Linear(patch_len, d_model)
        self.pos_emb   = nn.Parameter(torch.zeros(1, self.n_patches, d_model))
        nn.init.trunc_normal_(self.pos_emb, std=0.02)

    def forward(self, x):
        # x : (B, seq_len)
        B, L = x.shape
        # pad end with last value (paper's strategy)
        pad_len = self.stride
        x = torch.cat([x, x[:, -1:].expand(-1, pad_len)], dim=1)
        # unfold into patches : (B, n_patches, patch_len)
        x = x.unfold(dimension=1, size=self.patch_len, step=self.stride)
        # project + add positional embedding
        x = self.proj(x) + self.pos_emb[:, :x.size(1), :]
        return x   # (B, N, d_model)


class PatchTST(nn.Module):
    """
    Channel-independent PatchTST.

    forward() has two modes controlled by `mode`:
        'forecast'  : returns predictions  (B, pred_len, C)
        'pretrain'  : returns reconstructed masked patches  (B, N, patch_len)
    """
    def __init__(
        self,
        n_channels : int,
        seq_len    : int,
        pred_len   : int,
        patch_len  : int = 16,
        stride     : int = 8,
        d_model    : int = 128,
        n_heads    : int = 16,
        n_layers   : int = 3,
        d_ff       : int = 256,
        dropout    : float = 0.2,
    ):
        super().__init__()
        self.n_channels = n_channels
        self.pred_len   = pred_len
        self.patch_len  = patch_len

        self.patch_embed = PatchEmbedding(patch_len, stride, d_model, seq_len)
        N = self.patch_embed.n_patches

        encoder_layer = nn.TransformerEncoderLayer(
            d_model         = d_model,
            nhead           = n_heads,
            dim_feedforward = d_ff,
            dropout         = dropout,
            batch_first     = True,
            norm_first      = False,   # post-norm (BatchNorm applied separately)
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)

        # BatchNorm after attention (paper uses BN over LayerNorm)
        self.bn = nn.BatchNorm1d(d_model)

        # --- supervised head ---
        self.forecast_head = nn.Linear(N * d_model, pred_len)

        # --- self-supervised (masked autoencoder) head ---
        self.pretrain_head = nn.Linear(d_model, patch_len)

        self.d_model = d_model
        self.N       = N

    # ── instance norm (paper's RevIN-lite) ──────────────────────
    def _instance_norm(self, x):
        # x : (B, L, C)
        mean = x.mean(dim=1, keepdim=True)
        std  = x.std(dim=1, keepdim=True) + 1e-5
        return (x - mean) / std, mean, std

    def _instance_denorm(self, x, mean, std):
        # x : (B, pred_len, C)
        return x * std[:, :, :] + mean[:, :, :]

    # ── shared backbone (channel-independent) ───────────────────
    def _encode(self, x):
        """
        x : (B, seq_len, C)
        returns z : (B*C, N, d_model)
        """
        B, L, C = x.shape
        # reshape to treat each channel independently
        x = x.permute(0, 2, 1).reshape(B * C, L)     # (B*C, L)
        z = self.patch_embed(x)                        # (B*C, N, d_model)
        z = self.transformer(z)                        # (B*C, N, d_model)
        # apply BN over d_model dimension
        z = self.bn(z.reshape(-1, self.d_model)).reshape(B * C, self.N, self.d_model)
        return z, B, C

    # ── supervised forecast ──────────────────────────────────────
    def forecast(self, x):
        x, mean, std = self._instance_norm(x)
        z, B, C = self._encode(x)                     # (B*C, N, d_model)
        z = z.reshape(B * C, -1)                      # (B*C, N*d_model)
        out = self.forecast_head(z)                   # (B*C, pred_len)
        out = out.reshape(B, C, self.pred_len)        # (B, C, pred_len)
        out = out.permute(0, 2, 1)                    # (B, pred_len, C)
        out = self._instance_denorm(out, mean, std)
        return out

    # ── masked self-supervised pretraining ──────────────────────
    def pretrain(self, x, mask_ratio=0.4):
        """
        Returns (loss, reconstructed patches, mask).
        mask=1 means the patch was masked.
        """
        x, _, _ = self._instance_norm(x)
        B, L, C = x.shape
        xr = x.permute(0, 2, 1).reshape(B * C, L)    # (B*C, L)

        # PAD FIRST just like in PatchEmbedding
        xpad = torch.cat([xr, xr[:, -1:].expand(-1, self.patch_embed.stride)], dim=1)

        patches = self.patch_embed.proj(
            xpad.unfold(1, self.patch_len, self.patch_embed.stride)
        )                                              # (B*C, N, d_model) — no pos emb yet

        # build mask
        BC, N, _ = patches.shape
        n_mask   = int(N * mask_ratio)
        noise    = torch.rand(BC, N, device=x.device)
        ids_sort = noise.argsort(dim=1)
        mask     = torch.zeros(BC, N, device=x.device)
        mask.scatter_(1, ids_sort[:, :n_mask], 1)     # 1 = masked

        # zero out masked patches + add pos emb
        pos  = self.patch_embed.pos_emb[:, :N, :]
        inp  = patches * (1 - mask.unsqueeze(-1)) + pos
        z    = self.transformer(inp)
        z    = self.bn(z.reshape(-1, self.d_model)).reshape(BC, N, self.d_model)
        recon = self.pretrain_head(z)                 # (B*C, N, patch_len)

        # get target patches from raw series
        target = xpad.unfold(1, self.patch_len, self.patch_embed.stride)  # (B*C, N, P)

        # MSE only on masked patches
        loss = ((recon - target) ** 2 * mask.unsqueeze(-1)).sum() / (mask.sum() * self.patch_len + 1e-8)
        return loss

    def forward(self, x, mode='forecast'):
        if mode == 'forecast':
            return self.forecast(x)
        elif mode == 'pretrain':
            return self.pretrain(x)
        else:
            raise ValueError(f"Unknown mode: {mode}")

In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 4: Training utilities
# ─────────────────────────────────────────────────────────────────
def train_one_epoch(model, loader, optimizer, mode='forecast'):
    model.train()
    total_loss = 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        if mode == 'forecast':
            pred = model(x, mode='forecast')
            loss = F.mse_loss(pred, y)
        else:
            loss = model(x, mode='pretrain')
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    mse_total, mae_total, count = 0, 0, 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        pred = model(x, mode='forecast')
        mse_total += F.mse_loss(pred, y, reduction='sum').item()
        mae_total += F.l1_loss(pred, y, reduction='sum').item()
        count     += y.numel()
    return mse_total / count, mae_total / count


def run_training(model, train_loader, val_loader, n_epochs, lr, mode='forecast', desc=''):
    optimizer = Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max=n_epochs)
    best_val, best_state = float('inf'), None

    for epoch in range(1, n_epochs + 1):
        tr_loss = train_one_epoch(model, train_loader, optimizer, mode=mode)
        scheduler.step()

        if mode == 'forecast':
            val_mse, val_mae = evaluate(model, val_loader)
            if val_mse < best_val:
                best_val  = val_mse
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            if epoch % 10 == 0 or epoch == 1:
                print(f"[{desc}] epoch {epoch:3d}/{n_epochs} | "
                      f"train_loss={tr_loss:.4f} | val_MSE={val_mse:.4f} | val_MAE={val_mae:.4f}")
        else:
            if epoch % 10 == 0 or epoch == 1:
                print(f"[{desc}] epoch {epoch:3d}/{n_epochs} | pretrain_loss={tr_loss:.4f}")

    if best_state is not None:
        model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})
    return model

In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 5: Linear probing wrapper
# Freezes the backbone; only the forecast head is trained.
# ─────────────────────────────────────────────────────────────────
def freeze_backbone(model):
    """Freeze everything except the forecast head."""
    for name, param in model.named_parameters():
        if 'forecast_head' not in name:
            param.requires_grad = False

def unfreeze_all(model):
    for param in model.parameters():
        param.requires_grad = True

In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 6: Full linear-probing pipeline
#
# Steps (matching paper Table 4):
#   1. Self-supervised pretraining  (100 epochs)
#   2. Linear probing               (20 epochs, backbone frozen)
#   3. End-to-end fine-tuning       (20 epochs, full model)
#   4. Evaluate on test set → report MSE / MAE
# ─────────────────────────────────────────────────────────────────
def run_linear_probing_experiment(
    train_data, val_data, test_data,
    n_channels,
    seq_len    = 512,
    pred_len   = 96,
    patch_len  = 12,
    stride     = 12,     # non-overlapping for pretraining (paper sec 4.2)
    d_model    = 128,
    n_heads    = 16,
    n_layers   = 3,
    d_ff       = 256,
    dropout    = 0.2,
    batch_size = 32,
    pretrain_epochs   = 100,
    linprobe_epochs   = 20,
    finetune_epochs   = 20,
    pretrain_lr       = 1e-4,
    linprobe_lr       = 1e-4,
    finetune_lr       = 1e-5,
    mask_ratio        = 0.4,
    dataset_name      = 'dataset',
):
    print(f"\n{'='*60}")
    print(f"  Dataset: {dataset_name} | pred_len={pred_len} | channels={n_channels}")
    print(f"{'='*60}")

    # ── dataloaders ─────────────────────────────────────────────
    tr_loader, val_loader, te_loader = make_loaders(
        train_data, val_data, test_data,
        seq_len=seq_len, pred_len=pred_len, batch_size=batch_size
    )

    # ── build model ─────────────────────────────────────────────
    model = PatchTST(
        n_channels = n_channels,
        seq_len    = seq_len,
        pred_len   = pred_len,
        patch_len  = patch_len,
        stride     = stride,
        d_model    = d_model,
        n_heads    = n_heads,
        n_layers   = n_layers,
        d_ff       = d_ff,
        dropout    = dropout,
    ).to(DEVICE)

    total_params = sum(p.numel() for p in model.parameters())
    print(f"  Model params: {total_params:,}")

    # ── STEP 1: self-supervised pretraining ─────────────────────
    print(f"\n--- Step 1: Pretraining ({pretrain_epochs} epochs) ---")
    optimizer_pre = Adam(model.parameters(), lr=pretrain_lr, weight_decay=1e-4)
    scheduler_pre = CosineAnnealingLR(optimizer_pre, T_max=pretrain_epochs)

    model.train()
    for epoch in range(1, pretrain_epochs + 1):
        total_loss = 0
        for x, _ in tr_loader:
            x = x.to(DEVICE)
            optimizer_pre.zero_grad()
            loss = model(x, mode='pretrain')
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer_pre.step()
            total_loss += loss.item()
        scheduler_pre.step()
        if epoch % 20 == 0 or epoch == 1:
            print(f"  [Pretrain] epoch {epoch:3d}/{pretrain_epochs} | loss={total_loss/len(tr_loader):.4f}")

    # ── STEP 2: linear probing (freeze backbone) ─────────────────
    print(f"\n--- Step 2: Linear probing ({linprobe_epochs} epochs) ---")
    freeze_backbone(model)

    run_training(
        model, tr_loader, val_loader,
        n_epochs=linprobe_epochs, lr=linprobe_lr,
        mode='forecast', desc='LinProbe'
    )

    # ── STEP 3: end-to-end fine-tuning ───────────────────────────
    print(f"\n--- Step 3: Fine-tuning ({finetune_epochs} epochs) ---")
    unfreeze_all(model)

    run_training(
        model, tr_loader, val_loader,
        n_epochs=finetune_epochs, lr=finetune_lr,
        mode='forecast', desc='FineTune'
    )

    # ── STEP 4: test evaluation ───────────────────────────────────
    test_mse, test_mae = evaluate(model, te_loader)
    print(f"\n  ✓  Test  MSE={test_mse:.4f}  MAE={test_mae:.4f}")
    print(f"  (Paper Table 4 reference — pred_len={pred_len}: "
          f"Weather≈0.144/0.193, Electricity≈0.126/0.221)")

    return model, test_mse, test_mae

In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 7: Run all prediction horizons for Weather & Electricity
# Matches Table 4: T ∈ {96, 192, 336, 720}
# ─────────────────────────────────────────────────────────────────
PRED_LENS = [96, 192, 336, 720]
results   = {}

for pred_len in PRED_LENS:
    _, mse, mae = run_linear_probing_experiment(
        train_data   = w_train,
        val_data     = w_val,
        test_data    = w_test,
        n_channels   = w_train.shape[1],
        pred_len     = pred_len,
        dataset_name = f'Weather',
    )
    results[('Weather', pred_len)] = (mse, mae)

for pred_len in PRED_LENS:
    _, mse, mae = run_linear_probing_experiment(
        train_data   = e_train,
        val_data     = e_val,
        test_data    = e_test,
        n_channels   = e_train.shape[1],
        pred_len     = pred_len,
        # paper uses smaller model for larger datasets implicitly via d_model
        # electricity has many channels so keep d_model=128
        dataset_name = f'Electricity',
    )
    results[('Electricity', pred_len)] = (mse, mae)

# ── pretty summary table ──────────────────────────────────────────
print("\n" + "="*55)
print(f"{'Dataset':<14} {'T':>5}  {'MSE':>8}  {'MAE':>8}")
print("="*55)
for (dataset, pred_len), (mse, mae) in results.items():
    print(f"{dataset:<14} {pred_len:>5}  {mse:>8.4f}  {mae:>8.4f}")
print("="*55)


  Dataset: Weather | pred_len=96 | channels=21
  Model params: 934,892

--- Step 1: Pretraining (100 epochs) ---
  [Pretrain] epoch   1/100 | loss=0.9567
  [Pretrain] epoch  20/100 | loss=0.2083
  [Pretrain] epoch  40/100 | loss=0.2028
  [Pretrain] epoch  60/100 | loss=0.2001
  [Pretrain] epoch  80/100 | loss=0.1974
  [Pretrain] epoch 100/100 | loss=0.1962

--- Step 2: Linear probing (20 epochs) ---
[LinProbe] epoch   1/20 | train_loss=0.3799 | val_MSE=0.2971 | val_MAE=0.3155
[LinProbe] epoch  10/20 | train_loss=0.3568 | val_MSE=0.2900 | val_MAE=0.3089
[LinProbe] epoch  20/20 | train_loss=0.3556 | val_MSE=0.2895 | val_MAE=0.3078

--- Step 3: Fine-tuning (20 epochs) ---
[FineTune] epoch   1/20 | train_loss=0.3477 | val_MSE=0.2822 | val_MAE=0.3020
[FineTune] epoch  10/20 | train_loss=0.3242 | val_MSE=0.2743 | val_MAE=0.2937
[FineTune] epoch  20/20 | train_loss=0.3210 | val_MSE=0.2742 | val_MAE=0.2932

  ✓  Test  MSE=0.1784  MAE=0.2342
  (Paper Table 4 reference — pred_len=96: Weather≈0.

In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL: Table 5 — Transfer Learning (Linear Probing only)
#
# Pipeline:
#   1. Pretrain backbone on Electricity (same as before)
#   2. Swap forecast head for Weather's pred_len
#   3. Freeze backbone, train ONLY forecast head on Weather
#   4. Evaluate on Weather test set
# ─────────────────────────────────────────────────────────────────

def run_transfer_linprobe(
    # source (pretrain)
    src_train, src_val,
    src_channels,
    # target (linear probe)
    tgt_train, tgt_val, tgt_test,
    tgt_channels,
    pred_len,
    # architecture
    seq_len         = 512,
    patch_len       = 12,
    stride          = 12,
    d_model         = 128,
    n_heads         = 16,
    n_layers        = 3,
    d_ff            = 256,
    dropout         = 0.2,
    # training
    batch_size      = 32,
    pretrain_epochs = 100,
    linprobe_epochs = 20,
    pretrain_lr     = 1e-4,
    linprobe_lr     = 1e-4,
    src_name        = 'Electricity',
    tgt_name        = 'Weather',
):
    print(f"\n{'='*60}")
    print(f"  Transfer LinProbe | pretrain={src_name} → target={tgt_name}")
    print(f"  pred_len={pred_len} | src_C={src_channels} | tgt_C={tgt_channels}")
    print(f"{'='*60}")

    # ── source loaders (pretrain, no pred_len needed) ────────────
    # We reuse seq_len=512 windows; y is ignored during pretraining
    src_tr_loader, src_val_loader, _ = make_loaders(
        src_train, src_val, src_val,   # dummy test = val, unused
        seq_len=seq_len, pred_len=pred_len, batch_size=batch_size
    )

    # ── target loaders ───────────────────────────────────────────
    tgt_tr_loader, tgt_val_loader, tgt_te_loader = make_loaders(
        tgt_train, tgt_val, tgt_test,
        seq_len=seq_len, pred_len=pred_len, batch_size=batch_size
    )

    # ── build model on SOURCE channel count ──────────────────────
    model = PatchTST(
        n_channels = src_channels,   # electricity channels during pretrain
        seq_len    = seq_len,
        pred_len   = pred_len,
        patch_len  = patch_len,
        stride     = stride,
        d_model    = d_model,
        n_heads    = n_heads,
        n_layers   = n_layers,
        d_ff       = d_ff,
        dropout    = dropout,
    ).to(DEVICE)
    print(f"  Pretrain model params: {sum(p.numel() for p in model.parameters()):,}")

    # ── step 1: pretrain on Electricity ──────────────────────────
    print(f"\n[1/2] Pretraining on {src_name}  ({pretrain_epochs} epochs)")
    fit(model, src_tr_loader, src_val_loader,
        epochs=pretrain_epochs, lr=pretrain_lr,
        mode='pretrain', tag='Pre')

    # ── swap forecast head for target channel count ───────────────
    # The backbone (patch_embed + transformer + norm) is channel-independent
    # so weights transfer directly regardless of n_channels.
    # Only the forecast head changes shape.
    N = model.N
    model.n_channels   = tgt_channels
    model.forecast_head = nn.Linear(N * d_model, pred_len).to(DEVICE)
    print(f"\n  Swapped forecast head → output shape: ({N * d_model}, {pred_len})")

    # ── step 2: linear probing on Weather ────────────────────────
    # Freeze everything except the new forecast head
    print(f"\n[2/2] Linear probing on {tgt_name}  ({linprobe_epochs} epochs, backbone frozen)")
    freeze_backbone(model)

    # confirm what is trainable
    trainable = [n for n, p in model.named_parameters() if p.requires_grad]
    print(f"  Trainable params: {trainable}")

    fit(model, tgt_tr_loader, tgt_val_loader,
        epochs=linprobe_epochs, lr=linprobe_lr,
        mode='forecast', tag='LinProbe')

    # ── evaluate on target test set ───────────────────────────────
    test_mse, test_mae = evaluate(model, tgt_te_loader)
    print(f"\n  ► Test  MSE={test_mse:.4f}  MAE={test_mae:.4f}")
    return model, test_mse, test_mae

In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL: Run Table 5 — Lin. Prob for all 4 horizons
# ─────────────────────────────────────────────────────────────────
PRED_LENS = [96, 192, 336, 720]
transfer_linprobe_results = {}

for T in PRED_LENS:
    _, mse, mae = run_transfer_linprobe(
        src_train    = e_train,
        src_val      = e_val,
        src_channels = e_train.shape[1],   # 90

        tgt_train    = w_train,
        tgt_val      = w_val,
        tgt_test     = w_test,
        tgt_channels = w_train.shape[1],   # 21

        pred_len     = T,
    )
    transfer_linprobe_results[T] = (round(mse, 4), round(mae, 4))

# ── summary table vs Table 5 paper numbers ───────────────────────
paper_table5_linprobe = {
    96:  (0.163, 0.216),
    192: (0.205, 0.252),
    336: (0.253, 0.289),
    720: (0.320, 0.336),
}

print("\n" + "="*62)
print(f"  Table 5 — Transfer Learning → Weather  (Lin. Prob)")
print("="*62)
print(f"{'T':>5}  {'MSE':>8}  {'MAE':>8}  {'Paper MSE':>10}  {'Paper MAE':>10}")
print("-"*62)
for T in PRED_LENS:
    mse, mae   = transfer_linprobe_results[T]
    pmse, pmae = paper_table5_linprobe[T]
    print(f"{T:>5}  {mse:>8.4f}  {mae:>8.4f}  {pmse:>10.3f}  {pmae:>10.3f}")
print("="*62)